# Download Dataset

In [ ]:
import kagglehub

# Download latest version
dbdmobile_myanimelist_dataset_path = kagglehub.dataset_download("dbdmobile/myanimelist-dataset")

print(f"Import complete. Dataset saved at {dbdmobile_myanimelist_dataset_path}")

/opt/anaconda3/envs/anime/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 1.80G/1.80G [00:48<00:00, 39.7MB/s]

Extracting files...


Import complete. Dataset saved at /Users/marleybyers/.cache/kagglehub/datasets/dbdmobile/myanimelist-dataset/versions/5


In [ ]:
### Basic libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# Data Preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder

# Model Training
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

In [ ]:
# Load the dataset
df=pd.read_csv(f'{dbdmobile_myanimelist_dataset_path}/users-score-2023.csv', usecols=["user_id","anime_id","rating"])
print("Shape of the Dataset:",df.shape)
df.head()

Shape of the Dataset: (24325191, 3)


,user_id,anime_id,rating
0,1,21,9
1,1,48,7
2,1,320,5
3,1,49,8
4,1,304,8


In [ ]:
# Calculating the average score
avg_score = np.mean(df['rating'])
print('Average Score:', avg_score)

Average Score: 7.622930072779285


In [20]:
# Scaling our "rating" column
# Create a MinMaxScaler object
scaler = MinMaxScaler(feature_range=(0, 1))

# Scale the 'score' column between 0 and 1
df['scaled_score'] = scaler.fit_transform(df[['rating']])

In [108]:
# Encoding categorical data

## Encoding user IDs
user_encoder = LabelEncoder()
df["user_encoded"] = user_encoder.fit_transform(df["user_id"])
num_users = len(user_encoder.classes_)

## Encoding anime IDs
anime_encoder = LabelEncoder()
df["anime_encoded"] = anime_encoder.fit_transform(df["anime_id"])
num_animes = len(anime_encoder.classes_)

# Printing dataset information
print("Number of unique users: {}, Number of unique anime: {}".format(num_users, num_animes))
print("Minimum rating: {}, Maximum rating: {}".format(min(df['rating']), max(df['rating'])))

Number of unique users: 270033, Number of unique anime: 16500
Minimum rating: 1, Maximum rating: 10


In [48]:
# Shuffle the dataset
df = shuffle(df, random_state=100)

# Create feature matrix X and target variable y
X = df[['user_encoded', 'anime_encoded']].values
y = df["scaled_score"].values

# Printing dataset information
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (24325191, 2)
Shape of y: (24325191,)


In [27]:
test_set_size = 10000  # Number of samples to include in the test set

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_set_size, random_state=73)

print("Number of samples in the training set:", len(y_train))
print("Number of samples in the test set:", len(y_test))

Number of samples in the training set: 24315191
Number of samples in the test set: 10000


In [28]:
# X_train / X_test are (N, 2) int64 matrices: [user_encoded, anime_encoded]

In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Create custom Dataset class
class AnimeDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create DataLoader objects
train_dataset = AnimeDataset(X_train_tensor, y_train_tensor)
test_dataset = AnimeDataset(X_test_tensor, y_test_tensor)

batch_size = 10000
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
class RecommenderNet(nn.Module):
    """MLP on concatenated user/anime embeddings with scalar biases (NCF-style)."""

    def __init__(self, num_users, num_animes, embedding_size=128, dropout=0.2):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_size)
        self.anime_embedding = nn.Embedding(num_animes, embedding_size)
        self.user_bias = nn.Embedding(num_users, 1)
        self.anime_bias = nn.Embedding(num_animes, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))

        self.fc1 = nn.Linear(embedding_size * 2, 64)
        self.fc2 = nn.Linear(64, 1)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, user_ids, anime_ids):
        u = self.user_embedding(user_ids)
        v = self.anime_embedding(anime_ids)
        x = torch.cat([u, v], dim=1)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x) + self.user_bias(user_ids) + self.anime_bias(anime_ids) + self.global_bias
        return self.sigmoid(x)

In [124]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model = RecommenderNet(num_users, num_animes, embedding_size=128).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

In [114]:
def train_model(model, train_loader, criterion, optimizer, num_epochs=1):
    model.train()
    train_losses = []

    for epoch in range(num_epochs):
        running_loss = 0.0
        for user_anime, ratings in train_loader:
            user_ids = user_anime[:, 0].to(device)
            anime_ids = user_anime[:, 1].to(device)
            ratings = ratings.to(device)

            outputs = model(user_ids, anime_ids)
            loss = criterion(outputs.squeeze(), ratings)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        train_losses.append(avg_loss)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.6f}")

    return train_losses


In [115]:
def evaluate_model(model, test_loader, criterion):
    model.eval()
    test_loss = 0.0
    sse = 0.0
    n = 0

    with torch.no_grad():
        for user_anime, ratings in test_loader:
            user_ids = user_anime[:, 0].to(device)
            anime_ids = user_anime[:, 1].to(device)
            ratings = ratings.to(device)

            outputs = model(user_ids, anime_ids).squeeze()
            loss = criterion(outputs, ratings)
            test_loss += loss.item()
            sse += torch.sum((outputs - ratings) ** 2).item()
            n += ratings.numel()

    avg_loss = test_loss / len(test_loader)
    rmse = (sse / n) ** 0.5
    print(f"Test MSE: {avg_loss:.6f}  |  RMSE (0-1 scale): {rmse:.6f}")
    return avg_loss


In [ ]:
num_epochs = 3
train_losses = train_model(model, train_loader, criterion, optimizer, num_epochs)
test_loss = evaluate_model(model, test_loader, criterion)

# Plot the loss
plt.plot(train_losses, label="Training loss")
plt.title("Loss curve")
plt.xlabel("Epochs")
plt.ylabel("MSE")
plt.legend()
plt.show()


user_ids shape: torch.Size([10000])
anime_ids shape: torch.Size([10000])
Embedding(270033, 128)
torch.Size([10000, 128])
user_ids shape: torch.Size([10000])
anime_ids shape: torch.Size([10000])
Embedding(270033, 128)
torch.Size([10000, 128])
user_ids shape: torch.Size([10000])
anime_ids shape: torch.Size([10000])
Embedding(270033, 128)
torch.Size([10000, 128])
user_ids shape: torch.Size([10000])
anime_ids shape: torch.Size([10000])
Embedding(270033, 128)
torch.Size([10000, 128])
user_ids shape: torch.Size([10000])
anime_ids shape: torch.Size([10000])
Embedding(270033, 128)
torch.Size([10000, 128])
user_ids shape: torch.Size([10000])
anime_ids shape: torch.Size([10000])
Embedding(270033, 128)
torch.Size([10000, 128])
user_ids shape: torch.Size([10000])
anime_ids shape: torch.Size([10000])
Embedding(270033, 128)
torch.Size([10000, 128])
user_ids shape: torch.Size([10000])
anime_ids shape: torch.Size([10000])
Embedding(270033, 128)
torch.Size([10000, 128])
user_ids shape: torch.Size([1000

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x105e1f750>>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/anime/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


user_ids shape: torch.Size([10000])
anime_ids shape: torch.Size([10000])
Embedding(270033, 128)
torch.Size([10000, 128])
user_ids shape: torch.Size([10000])
anime_ids shape: torch.Size([10000])
Embedding(270033, 128)
torch.Size([10000, 128])
user_ids shape: torch.Size([10000])
anime_ids shape: torch.Size([10000])
Embedding(270033, 128)
torch.Size([10000, 128])
user_ids shape: torch.Size([10000])
anime_ids shape: torch.Size([10000])
Embedding(270033, 128)
torch.Size([10000, 128])
user_ids shape: torch.Size([10000])
anime_ids shape: torch.Size([10000])
Embedding(270033, 128)
torch.Size([10000, 128])
user_ids shape: torch.Size([10000])
anime_ids shape: torch.Size([10000])
Embedding(270033, 128)
torch.Size([10000, 128])
user_ids shape: torch.Size([10000])
anime_ids shape: torch.Size([10000])
Embedding(270033, 128)
torch.Size([10000, 128])
user_ids shape: torch.Size([10000])
anime_ids shape: torch.Size([10000])
Embedding(270033, 128)
torch.Size([10000, 128])
user_ids shape: torch.Size([1000